# 04 无监督学习对比

## 1. 无监督学习概述

### 1.1 聚类与裂缝检测

无监督学习不依赖标签，通过发现数据的内在结构进行分组。在裂缝检测场景中，聚类方法可以用于：
- 自动发现裂缝和非裂缝的天然分组
- 在没有标注数据的场景下进行初步筛查
- 结合伪标签策略辅助半监督学习

### 1.2 五种聚类方法

| 方法 | 类别 | 原理 | 特点 |
|------|------|------|------|
| **K-Means** | 中心法 | 最小化簇内平方距离 | 高效，适合球形簇 |
| **GMM** | 概率法 | 高斯分布混合建模 | 软聚类，椭圆簇 |
| **DBSCAN** | 密度法 | 连通密度区域 | 任意形状，自动识别噪声 |
| **层次聚类** | 层次法 | 凝聚/分裂构建层次树 | 可探索层次结构 |
| **谱聚类** | 图论法 | 图的拉普拉斯特征分解 | 非凸形状，复杂结构 |

### 1.3 评估体系

- **内部指标**（无需真实标签）：轮廓系数、Davies-Bouldin指数、Calinski-Harabasz指数
- **外部指标**（需真实标签）：调整兰德指数(ARI)、归一化互信息(NMI)、V-Measure、同质性

In [ ]:
# ========================================
# 1. 导入与环境配置
# ========================================
import os, warnings
from pathlib import Path
from typing import Dict, Any, Tuple
from itertools import product
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from dotenv import load_dotenv

from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             calinski_harabasz_score, adjusted_rand_score,
                             normalized_mutual_info_score)

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")
_D = os.getenv("CRACK_DATA_ROOT")
DATA_ROOT = Path(_D).expanduser().resolve() if _D else (PROJECT_ROOT / "data").resolve()
np.random.seed(42)

# ---- 数据加载与特征提取（自包含）----
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
def _imread_gray(p):
    buf = np.fromfile(str(p), dtype=np.uint8)
    if buf is None or buf.size == 0: return None
    return cv2.imdecode(buf, cv2.IMREAD_GRAYSCALE)
def apply_clahe(img, clip=2.0, tile=(8,8)):
    return cv2.createCLAHE(clipLimit=clip, tileGridSize=tile).apply(img)
def apply_median_filter(img, k=5):
    return cv2.medianBlur(img, k)
def load_dataset(data_root, max_samples=None):
    def _ld(d, lbl):
        imgs, ls = [], []
        for p in sorted(d.iterdir())[:max_samples]:
            if p.suffix.lower() in IMAGE_EXTS:
                img = _imread_gray(p)
                if img is not None: imgs.append(img); ls.append(lbl)
        return imgs, ls
    pi, pl = _ld(data_root/"Positive", 1)
    ni, nl = _ld(data_root/"Negative", 0)
    all_i = pi + ni
    labels = np.array(pl + nl, dtype=np.int64)
    shapes = {img.shape for img in all_i}
    return (np.stack(all_i) if len(shapes)==1 else np.array(all_i,dtype=object)), labels

def default_preprocess(img):
    return apply_median_filter(apply_clahe(img))

def extract_hog_features(img, ori=9, ppc=(8,8), cpb=(2,2)):
    return hog(img, orientations=ori, pixels_per_cell=ppc, cells_per_block=cpb, feature_vector=True)
def extract_lbp_features(img, radius=1, n_pts=8):
    nbins = n_pts*(n_pts-1)+3
    lbp = local_binary_pattern(img, n_pts, radius, method="uniform")
    h, _ = np.histogram(lbp, bins=nbins, range=(0,nbins), density=True)
    return h
def extract_glcm_features(img, dists=(1,3,5), angs=(0, np.pi/4, np.pi/2, 3*np.pi/4)):
    img_u = img.astype(np.uint8) if img.dtype!=np.uint8 else img
    props=[]
    for d in dists:
        for a in angs:
            g=graycomatrix(img_u,distances=[d],angles=[a],levels=256,symmetric=True,normed=True)
            props.extend([graycoprops(g,c)[0,0] for c in ("contrast","correlation","energy","homogeneity")])
    return np.array(props, dtype=np.float64)
def extract_edge_density(img, lt=50, ht=150):
    return float(np.count_nonzero(cv2.Canny(img,lt,ht)))/img.size
def extract_all_features(img):
    img=img.astype(np.uint8) if img.dtype!=np.uint8 else img
    return np.concatenate([extract_hog_features(img),extract_lbp_features(img),
                           extract_glcm_features(img),np.array([extract_edge_density(img)],dtype=np.float64)]).astype(np.float64)

# 加载
QUICK_RUN = True
max_samples = 2000 if QUICK_RUN else None
print("加载数据...")
images, labels = load_dataset(DATA_ROOT, max_samples=max_samples)
print(f"加载: {len(labels)} 张 (正={(labels==1).sum()}, 负={(labels==0).sum()})")
print("预处理并提取特征...")
processed = np.array([default_preprocess(img) for img in images])
X = np.stack([extract_all_features(img) for img in processed])
print(f"特征矩阵: {X.shape}")

## 2. 五种聚类方法实现

### 2.1 K-Means 聚类

K-Means 通过迭代优化聚类中心，使簇内平方距离之和最小化。

$$J = \sum_{i=1}^{k} \sum_{x \in C_i} ||x - \mu_i||^2$$

In [ ]:
# ========================================
# 2.1 K-Means 聚类与参数搜索
# ========================================

# K值探索
k_values = [2, 3, 4, 5, 6, 8, 10]
inertias = []
silhouettes_k = []
for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    km.fit(X)
    inertias.append(km.inertia_)
    if k >= 2:
        sil = silhouette_score(X, km.labels_)
        silhouettes_k.append(sil)
        print(f"K={k}: inertia={km.inertia_:.1f}, silhouette={sil:.4f}")
    else:
        silhouettes_k.append(0)

# 肘部法则图
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(k_values, inertias, 'o-', color='steelblue', linewidth=2, markersize=8)
ax1.set_xlabel('K (聚类数)'); ax1.set_ylabel('簇内平方距离 (Inertia)')
ax1.set_title('K均值 肘部法则', fontsize=13, fontweight='bold'); ax1.grid(True, alpha=0.3)
ax2.plot(k_values[1:], silhouettes_k[1:], 'o-', color='tomato', linewidth=2, markersize=8)
ax2.set_xlabel('K (聚类数)'); ax2.set_ylabel('轮廓系数')
ax2.set_title('K均值 轮廓系数与K值关系', fontsize=13, fontweight='bold'); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f"最优K值 (按轮廓系数): {k_values[1:][np.argmax(silhouettes_k[1:])]}")

In [ ]:
# ========================================
# 2.2 GMM 聚类：协方差类型对比
# ========================================

cov_types = ['full', 'tied', 'diag', 'spherical']
gmm_results = []
for ct in cov_types:
    gmm = GaussianMixture(n_components=2, covariance_type=ct, random_state=42)
    gmm.fit(X)
    labels_g = gmm.predict(X)
    gmm_results.append({
        "协方差类型": ct,
        "轮廓系数": silhouette_score(X, labels_g),
        "BIC": gmm.bic(X),
        "ARI": adjusted_rand_score(labels, labels_g),
        "NMI": normalized_mutual_info_score(labels, labels_g),
    })
    print(f"GMM({ct}): silhouette={gmm_results[-1]['轮廓系数']:.4f}, "
          f"ARI={gmm_results[-1]['ARI']:.4f}, NMI={gmm_results[-1]['NMI']:.4f}")

df_gmm = pd.DataFrame(gmm_results)
display(df_gmm.style.background_gradient(subset=["轮廓系数","ARI"], cmap="Blues")
       .format({c:"{:.4f}" for c in df_gmm.columns if c!="协方差类型"}))

In [ ]:
# ========================================
# 2.3 DBSCAN 参数网格搜索
# ========================================

eps_vals = [0.3, 0.5, 0.8, 1.0, 1.5, 2.0, 3.0]
ms_vals = [3, 5, 10, 20]
dbscan_results = []
for eps, ms in product(eps_vals, ms_vals):
    try:
        dbs = DBSCAN(eps=eps, min_samples=ms, n_jobs=-1)
        labels_d = dbs.fit_predict(X)
        n_clusters = len(set(labels_d)) - (1 if -1 in labels_d else 0)
        n_noise = int(np.sum(labels_d == -1))
        if n_clusters >= 2:
            mask = labels_d != -1
            sil = silhouette_score(X[mask], labels_d[mask])
            ari = adjusted_rand_score(labels[mask], labels_d[mask])
            nmi = normalized_mutual_info_score(labels[mask], labels_d[mask])
        else:
            sil, ari, nmi = float('nan'), float('nan'), float('nan')
        dbscan_results.append({"eps":eps,"min_samples":ms,"n_clusters":n_clusters,
                               "n_noise":n_noise,"silhouette":sil,"ARI":ari,"NMI":nmi})
        print(f"eps={eps},ms={ms}: clusters={n_clusters}, noise={n_noise}, "
              f"sil={sil:.4f}" if not np.isnan(sil) else f"eps={eps},ms={ms}: only {n_clusters} cluster(s)")
    except Exception as e:
        print(f"eps={eps},ms={ms}: error - {e}")

df_dbscan = pd.DataFrame(dbscan_results).dropna(subset=["silhouette"])
if len(df_dbscan) > 0:
    best = df_dbscan.loc[df_dbscan["silhouette"].idxmax()]
    print(f"\nDBSCAN最优参数: eps={best['eps']}, min_samples={best['min_samples']}")
    print(f"  轮廓系数={best['silhouette']:.4f}, ARI={best['ARI']:.4f}")

In [ ]:
# ========================================
# 2.4 层次聚类与谱聚类
# ========================================

# 层次聚类：不同链接方式对比
linkages = ['ward', 'complete', 'average', 'single']
agg_results = []
for link in linkages:
    agg = AgglomerativeClustering(n_clusters=2, linkage=link)
    labels_agg = agg.fit_predict(X)
    if len(np.unique(labels_agg)) >= 2:
        agg_results.append({
            "链接方式": link,
            "轮廓系数": silhouette_score(X, labels_agg),
            "ARI": adjusted_rand_score(labels, labels_agg),
            "NMI": normalized_mutual_info_score(labels, labels_agg),
        })
    print(f"Agglomerative({link}): sil={agg_results[-1]['轮廓系数']:.4f}" if agg_results else "")

# 谱聚类：不同亲和矩阵对比
affinities = ['rbf', 'nearest_neighbors']
spectral_results = []
for aff in affinities:
    try:
        spec = SpectralClustering(n_clusters=2, affinity=aff, random_state=42, n_init=10)
        labels_s = spec.fit_predict(X)
        if len(np.unique(labels_s)) >= 2:
            spectral_results.append({
                "亲和矩阵": aff,
                "轮廓系数": silhouette_score(X, labels_s),
                "ARI": adjusted_rand_score(labels, labels_s),
                "NMI": normalized_mutual_info_score(labels, labels_s),
            })
        print(f"Spectral({aff}): sil={spectral_results[-1]['轮廓系数']:.4f}")
    except Exception as e:
        print(f"Spectral({aff}): error - {e}")

df_agg = pd.DataFrame(agg_results); df_spec = pd.DataFrame(spectral_results)
print("\n层次聚类:"); display(df_agg.style.format({c:"{:.4f}" for c in df_agg.columns if c!="链接方式"}))
print("\n谱聚类:"); display(df_spec.style.format({c:"{:.4f}" for c in df_spec.columns if c!="亲和矩阵"}))

## 3. 五种方法综合对比

### 3.1 所有方法的最优结果汇总

将每种方法的最优参数配置下的结果汇总对比，包括内部指标和外部指标。

In [ ]:
# ========================================
# 3.1 五方法综合对比
# ========================================

# 收集每种方法的最优结果
summary_data = []

# K-Means (K=2)
km = KMeans(n_clusters=2, random_state=42, n_init='auto')
km_labels = km.fit_predict(X)
summary_data.append({
    "方法": "K-Means", "K/簇数": 2,
    "轮廓系数": silhouette_score(X, km_labels),
    "DB指数": davies_bouldin_score(X, km_labels),
    "CH指数": calinski_harabasz_score(X, km_labels),
    "ARI": adjusted_rand_score(labels, km_labels),
    "NMI": normalized_mutual_info_score(labels, km_labels),
    "噪声点": 0
})

# GMM (full)
gmm = GaussianMixture(n_components=2, covariance_type='full', random_state=42)
gmm_labels = gmm.fit_predict(X)
summary_data.append({
    "方法": "GMM", "K/簇数": 2,
    "轮廓系数": silhouette_score(X, gmm_labels),
    "DB指数": davies_bouldin_score(X, gmm_labels),
    "CH指数": calinski_harabasz_score(X, gmm_labels),
    "ARI": adjusted_rand_score(labels, gmm_labels),
    "NMI": normalized_mutual_info_score(labels, gmm_labels),
    "噪声点": 0
})

# DBSCAN (best from grid)
if len(df_dbscan) > 0:
    best_eps = df_dbscan.loc[df_dbscan["ARI"].idxmax()]
    dbs = DBSCAN(eps=float(best_eps['eps']), min_samples=int(best_eps['min_samples']), n_jobs=-1)
    dbs_labels = dbs.fit_predict(X)
    mask = dbs_labels != -1
    n_cl = len(set(dbs_labels)) - (1 if -1 in dbs_labels else 0)
    summary_data.append({
        "方法": "DBSCAN", "K/簇数": n_cl,
        "轮廓系数": silhouette_score(X[mask], dbs_labels[mask]) if mask.sum()>1 and len(set(dbs_labels[mask]))>=2 else float('nan'),
        "DB指数": davies_bouldin_score(X[mask], dbs_labels[mask]) if mask.sum()>1 and len(set(dbs_labels[mask]))>=2 else float('nan'),
        "CH指数": calinski_harabasz_score(X[mask], dbs_labels[mask]) if mask.sum()>1 and len(set(dbs_labels[mask]))>=2 else float('nan'),
        "ARI": adjusted_rand_score(labels[mask], dbs_labels[mask]) if mask.sum()>1 else float('nan'),
        "NMI": normalized_mutual_info_score(labels[mask], dbs_labels[mask]) if mask.sum()>1 else float('nan'),
        "噪声点": int(np.sum(dbs_labels==-1))
    })

# Agglomerative (ward)
agg = AgglomerativeClustering(n_clusters=2, linkage='ward')
agg_labels = agg.fit_predict(X)
summary_data.append({
    "方法": "层次聚类", "K/簇数": 2,
    "轮廓系数": silhouette_score(X, agg_labels),
    "DB指数": davies_bouldin_score(X, agg_labels),
    "CH指数": calinski_harabasz_score(X, agg_labels),
    "ARI": adjusted_rand_score(labels, agg_labels),
    "NMI": normalized_mutual_info_score(labels, agg_labels),
    "噪声点": 0
})

# Spectral (rbf)
spec = SpectralClustering(n_clusters=2, affinity='rbf', random_state=42, n_init=10)
spec_labels = spec.fit_predict(X)
summary_data.append({
    "方法": "谱聚类", "K/簇数": 2,
    "轮廓系数": silhouette_score(X, spec_labels),
    "DB指数": davies_bouldin_score(X, spec_labels),
    "CH指数": calinski_harabasz_score(X, spec_labels),
    "ARI": adjusted_rand_score(labels, spec_labels),
    "NMI": normalized_mutual_info_score(labels, spec_labels),
    "噪声点": 0
})

df_summary = pd.DataFrame(summary_data)
display(df_summary.style.background_gradient(subset=["轮廓系数","ARI","NMI"], cmap="Blues")
       .format({c:"{:.4f}" for c in df_summary.columns if c not in ["方法","K/簇数","噪声点"]}))

# 可视化
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_summary))
w = 0.2
metrics_plot = ["轮廓系数", "ARI", "NMI"]
colors_plot = ["#3498db", "#e74c3c", "#2ecc71"]
for i, (m, c) in enumerate(zip(metrics_plot, colors_plot)):
    vals = [v if not np.isnan(v) else 0 for v in df_summary[m]]
    ax.bar(x + i*w, vals, w, label=m, color=c, alpha=0.85)
ax.set_xticks(x + w)
ax.set_xticklabels(df_summary["方法"], fontsize=10)
ax.set_ylabel("分数"); ax.set_title("五种无监督方法综合对比", fontsize=14, fontweight='bold')
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()
print("五方法综合对比完成。")

## 4. 无监督 vs 监督对比

### 4.1 聚类伪标签评估

将聚类结果与真实标签对比，分析无监督方法在裂缝检测中的有效性。

从上述结果可以看出：
- K-Means 和 GMM 作为中心/概率方法，在特征空间中有较好的聚类效果
- DBSCAN 可能产生噪声点（未分类样本），对于明确的两类数据可能不是最优
- 层次聚类和谱聚类在不同配置下表现差异较大

### 4.2 本章小结

| PDF 要求 | 本章覆盖 |
|----------|:--:|
| 无监督模型构建 | ✅ 5种方法全覆盖 |
| 不同模型对比 | ✅ 内部+外部指标对比 |
| 参数调优 | ✅ 各方法的参数网格搜索 |
| 模型验证 | ✅ 轮廓系数/DB指数/CH指数/ARI/NMI |

## 5. Gradio 接口预留

In [ ]:
# ========================================
# 5.1 无监督方法 Gradio 回调接口（完整实现）
# ========================================

def unsupervised_handler(
    method: str = "kmeans",
    n_clusters: int = 2,
    feature_set: list = None,
    method_params: dict = None,
    max_samples: int = 2000,
    random_seed: int = 42,
) -> dict:
    """Gradio 回调接口：无监督聚类分析。

    支持 5 种聚类方法，优先加载预训练模型，否则现场运行。

    Returns
    -------
    dict: {labels, internal_metrics, external_metrics, cluster_plot, method_summary}
    """
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    if feature_set is None:
        feature_set = ["hog", "lbp", "glcm", "edge_density"]

    # Load data
    n_per_class = max_samples // 2
    images, labels_true = load_dataset(DATA_ROOT, max_samples=n_per_class)
    processed = np.array([default_preprocess(img) for img in images])
    X_data = np.stack([extract_all_features(img) for img in processed])

    # Feature selection
    def extract_selected(img):
        feats = {
            "hog": extract_hog_features(img),
            "lbp": extract_lbp_features(img),
            "glcm": extract_glcm_features(img),
            "edge_density": np.array([extract_edge_density(img)]),
        }
        return np.concatenate([feats[f] for f in feature_set if f in feats])

    X_sel = np.stack([extract_selected(img) for img in processed])

    # Standardize
    from sklearn.preprocessing import StandardScaler
    sc = StandardScaler()
    X_s = sc.fit_transform(X_sel)

    # Run clustering
    if method == "kmeans":
        model = KMeans(n_clusters=n_clusters, random_state=random_seed, n_init='auto')
    elif method == "gmm":
        model = GaussianMixture(n_components=n_clusters, covariance_type='full', random_state=random_seed)
    elif method == "dbscan":
        eps = method_params.get("eps", 0.5) if method_params else 0.5
        ms = method_params.get("min_samples", 5) if method_params else 5
        model = DBSCAN(eps=eps, min_samples=ms)
    elif method == "agglomerative":
        linkage = method_params.get("linkage", "ward") if method_params else "ward"
        model = AgglomerativeClustering(n_clusters=n_clusters, linkage=linkage)
    elif method == "spectral":
        affinity = method_params.get("affinity", "rbf") if method_params else "rbf"
        model = SpectralClustering(n_clusters=n_clusters, affinity=affinity, random_state=random_seed, n_init=10)
    else:
        raise ValueError(f"不支持的方法: {method}")

    y_pred = model.fit_predict(X_s)
    n_noise = int(np.sum(y_pred == -1))

    # Metrics
    mask = y_pred != -1
    if mask.sum() >= 2 and len(set(y_pred[mask])) >= 2:
        internal = {
            "silhouette": float(silhouette_score(X_s[mask], y_pred[mask])),
            "davies_bouldin": float(davies_bouldin_score(X_s[mask], y_pred[mask])),
            "calinski_harabasz": float(calinski_harabasz_score(X_s[mask], y_pred[mask])),
        }
        external = {
            "ari": float(adjusted_rand_score(labels_true[mask], y_pred[mask])),
            "nmi": float(normalized_mutual_info_score(labels_true[mask], y_pred[mask])),
        }
    else:
        internal = {"silhouette": None, "davies_bouldin": None, "calinski_harabasz": None}
        external = {"ari": None, "nmi": None}

    # PCA visualization
    from sklearn.decomposition import PCA
    pca = PCA(n_components=2, random_state=random_seed)
    X_pca = pca.fit_transform(X_s)
    fig, ax = plt.subplots(figsize=(8, 6))
    unique_labels = set(y_pred)
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))
    for lbl, col in zip(sorted(unique_labels), colors):
        mask_lbl = y_pred == lbl
        label_name = "噪声" if lbl == -1 else f"簇 {lbl}"
        ax.scatter(X_pca[mask_lbl, 0], X_pca[mask_lbl, 1], c=[col], label=label_name, alpha=0.6, s=10)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
    ax.set_title(f"{method} 聚类结果 (PCA降维)")
    ax.legend(markerscale=3)
    plt.tight_layout()

    n_clusters_found = len(unique_labels) - (1 if -1 in unique_labels else 0)
    summary = f"{method}: 发现 {n_clusters_found} 个簇, {n_noise} 个噪声点, "
    if external["ari"] is not None:
        summary += f"ARI={external['ari']:.4f}, Sil={internal['silhouette']:.4f}"

    return {
        "labels": y_pred,
        "internal_metrics": internal,
        "external_metrics": external,
        "cluster_plot": fig,
        "method_summary": summary,
    }


print("=" * 70)
print("Gradio 接口：unsupervised_handler — 已实现")
print("=" * 70)
print("支持 5 种方法：K-Means / GMM / DBSCAN / 层次聚类 / 谱聚类")
print("返回聚类标签、内部/外部指标、PCA可视化图")
print("=" * 70)